# 🎬 Bitrate Backtrack — 1440p Cloud Video Assembly Engine
### Automated Countdown Compilation with Free Nvidia T4 GPU (Google Colab)

This notebook completely solves the **slow home internet upload** and **VPS CPU freeze** problems:
1. **Pristine 1440p (2560x1440 @ 60fps):** Standardizes all video to 1440p to unlock YouTube's premium VP9/AV1 high-bitrate encoding.
2. **Zero Black Bars:** Automatically crops or ambiently blurs non-16:9 gameplay footage.
3. **Intro Trailer Montage:** Takes rapid cuts from trailers or gameplay clips and syncs them to your Intro Voiceover with audio ducking.
4. **Free Nvidia T4 GPU (16GB VRAM):** Video rendering uses hardware NVENC acceleration (`h264_nvenc`) — renders a 30-minute 1440p60 countdown in **~3-5 minutes**.
5. **Zero Local Bandwidth:** The finished 2-4 GB master video is saved directly to your **Google Drive** internally.

## Step 1: Verify Free Nvidia GPU & Mount Google Drive
Make sure your Colab runtime has a GPU: Go to **Runtime -> Change runtime type -> T4 GPU**.

In [ ]:
# Check GPU status
!nvidia-smi

# Mount your Google Drive to save the final video and load voiceovers
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Clone Repo & Install Requirements

In [ ]:
# Clone repository
!git clone https://github.com/Clarionxk/segment-dw.git /content/segment-dw
%cd /content/segment-dw

# Install dependencies and ensure FFmpeg is up to date
!pip install -r requirements.txt
!apt-get update -qq && apt-get install -y ffmpeg -qq

## Step 3: Load Voiceover Audio Files & Trailers
Place your VO files (`intro.mp3`, `01.mp3`, `02.mp3`, etc.) into `voiceovers`.
If you have a dedicated trailers folder, place them into `trailers/` (or the script will automatically sample your downloaded gameplay clips for the intro montage).

In [ ]:
import os, shutil
os.makedirs('voiceovers', exist_ok=True)
os.makedirs('trailers', exist_ok=True)

# Copy from Google Drive if you have folders named 'voiceovers' or 'trailers'
for folder in ['voiceovers', 'trailers']:
    drive_path = f'/content/drive/MyDrive/{folder}'
    if os.path.exists(drive_path):
        for f in os.listdir(drive_path):
            src = os.path.join(drive_path, f)
            if os.path.isfile(src):
                shutil.copy(src, os.path.join(folder, f))
        print(f"[+] Copied {folder} from Google Drive.")

!ls -la voiceovers

## Step 4: Download Gameplay Clips at Datacenter Speed (800+ Mbps)

In [ ]:
# Download all clips defined in videoplan1.md at 1080p or 1440p
!python download.py videoplan1.md --res 1080p

## Step 5: Assemble 1440p Master Video (No Black Bars, Audio Ducking & Intro Montage)
Options:
- `--target-res 1440p`: Native 2560x1440 @ 60fps (premium YouTube quality)
- `--aspect crop`: Smart center-fill to 16:9 edge-to-edge (no black bars, no stretch)
- `--aspect blur`: Frosted ambient blur background (for retro 4:3 clips, full HUD)
- `--intro-cut 3.5`: Rapid cuts every 3.5s across trailers for the intro VO

In [ ]:
# Assemble full countdown with GPU acceleration
!python assemble.py videoplan1.md --gpu --target-res 1440p --aspect crop --solo-sfx 10

## Step 6: Save Master Video Directly to Google Drive
Transfers internally inside Google's network in seconds — zero home internet bandwidth used!

In [ ]:
# Copy final master video and chapter markers directly to Google Drive
!cp output/master_compilation.mp4 /content/drive/MyDrive/master_compilation.mp4
!cp output/youtube_chapters.txt /content/drive/MyDrive/youtube_chapters.txt

print("\n🎉 SUCCESS! Your finished 1440p compilation video is saved in your Google Drive: MyDrive/master_compilation.mp4")
print("Chapter markers saved to: MyDrive/youtube_chapters.txt")